<a href="https://colab.research.google.com/github/RAJAMURUGAN-VS/genai-learning-journey/blob/main/04-rag/02_dynamic_pdf_rag_assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U langchain-community pypdf
!pip install -qU langchain-text-splitters
!pip install -qU langchain langchain-huggingface sentence_transformers
!pip install -U langchain-chroma

Build Vector Store from PDF URL

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

In [ ]:
def build_vector_store(pdf_url):

  #Load Document

  loader=PyPDFLoader(pdf_url)
  doc=loader.load()

  #Splitting Document

  text_splitter=RecursiveCharacterTextSplitter(
      chunk_size=1000,
      chunk_overlap=200,
      length_function=len,
  )

  all_splits=text_splitter.split_documents(doc)

  #Creating Embeddings

  embedding_model=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
  )

  vector_store=Chroma(
    collection_name="research_collection",
    embedding_function=embedding_model,
    persist_directory="./chroma_langchain_db"
  )

  vector_store.add_documents(documents=all_splits)

  return vector_store

Retrieve Context

In [ ]:
def retrieve_context(vector_store, query, k=5):
  retrieved_docs=vector_store.similarity_search(query, k=k)

  docs_content=""
  for doc in retrieved_docs:
    docs_content+=f"Source: {doc.metadata}\n"
    docs_content+=f"Content: {doc.page_content}\n\n"

  return docs_content, retrieved_docs

Model

In [ ]:
!pip install -U langchain-google-genai

In [ ]:
from langchain.chat_models import init_chat_model
from google.colab import userdata

In [ ]:
#Initializing The Model

api_key=userdata.get('GEMINI_API_KEY')
model=init_chat_model(
   "google_genai:gemini-2.5-flash",
   api_key=api_key,
)

Chat Function

In [ ]:
def docu_chat(pdf_url, user_query):

  vector_store=build_vector_store(pdf_url)

  context, source_docs=retrieve_context(vector_store, user_query, k=5)

  #Setting Up the LLM Instructions

  system_message=f"""
  You are a helpful chatbot.
  Use only the following pieces of context to answer the question.
  Don't makeup any new information.
  Context: {context}
  """

  messages=[
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_query}
  ]

  #Invoking LLM And Getting Results

  response=model.invoke(messages)

  sources = []

  for doc in source_docs:
    page = doc.metadata.get("page", 0) + 1
    sources.append(f"Page {page}")

  return (response.content+"\n\nSources:\n"+ "\n".join(sources))

  return response.content

Gradio UI

In [ ]:
!pip install -q gradio

In [ ]:
import gradio as gr

In [ ]:
demo = gr.Interface(
    fn=docu_chat,
    inputs=[
        gr.Textbox(
            label="PDF URL",
            placeholder="https://arxiv.org/pdf/1706.03762"
        ),
        gr.Textbox(
            lines=4,
            label="Question",
            placeholder="Ask a question about the document..."
        )
    ],
    outputs=gr.Textbox(
        lines=10,
        label="Answer"
    ),
    title="PDF RAG Assistant",
    description="Enter a PDF URL and ask questions about the document."
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cb71fbf47c68867594.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
